# phase 4: SQL Analysis
## London Airbnb Analysis - Mohan Raju Kolikonda

Three queries answering the three business questions from the project charter.
Data source: analysis-ready CSVs from Phase 3.

In [1]:
import pandas as pd 
import sqlite3

listings = pd.read_csv('../data/listings_clean.csv')
reviews = pd.read_csv('../data/reviews_clean.csv')

conn = sqlite3.connect('../data/airbnb.db')
listings.to_sql('listings',conn, if_exists='replace', index=False)
reviews.to_sql('reviews', conn, if_exists='replace', index=False)

print("Listings loaded:", listings.shape)
print("Reviews loaded:", reviews.shape)


Listings loaded: (92638, 31)
Reviews loaded: (978778, 5)


In [2]:
pd.read_sql("SELECT COUNT(*) AS n_listings FROM listings", conn)

,n_listings
0,92638


In [3]:
pd.read_sql("SELECT MAX(price) AS max_price, COUNT(*) AS n FROM listings", conn)

,max_price,n
0,1373.4575,92638


### Query 1 — Business Question 1

Which boroughs and room types command the highest nightly prices, and how large is the price gap between them?

Columns used: neighbourhood_cleansed, room_type, price.

In [4]:
query1 = """
SELECT
     neighbourhood_cleansed AS borough,
     room_type,
     COUNT(*) AS n_listings,
     ROUND(AVG(price), 2) AS avg_price,
     ROUND(MIN(price), 2) AS min_price,
     ROUND(MAX(price), 2) AS max_price
FROM listings
WHERE price IS NOT NULL
GROUP BY neighbourhood_cleansed, room_type
HAVING COUNT(*) >= 30
ORDER BY avg_price DESC
LIMIT 20
""" 
pd.read_sql(query1, conn)

,borough,room_type,n_listings,avg_price,min_price,max_price
0,Westminster,Entire home/apt,6892,450.79,7.03,1373.46
1,Kensington and Chelsea,Entire home/apt,4030,417.72,8.76,1373.46
2,City of London,Entire home/apt,410,392.55,59.62,1373.46
3,City of London,Private room,48,330.94,79.00,1373.46
4,Camden,Entire home/apt,3158,330.93,13.80,1373.46
5,Wandsworth,Entire home/apt,1925,326.18,25.85,1373.46
6,Richmond upon Thames,Entire home/apt,563,301.22,14.26,1373.46
7,Hammersmith and Fulham,Entire home/apt,2040,292.37,18.33,1373.46
8,Merton,Entire home/apt,667,288.82,42.82,1373.46
9,Lambeth,Entire home/apt,1784,276.77,25.48,1373.46


In [5]:
with open('../sql/query1_borough_room_type_prices.sql', 'w') as f:
    f.write(query1)
print("Saved query1.sql")

Saved query1.sql


### Query 2 — Business Question 2

Do listings managed by superhosts achieve higher review scores and estimated booking activity than non-superhost listings?

Columns used: host_is_superhost, review_scores_rating, number_of_reviews, reviews_per_month, neighbourhood_cleansed.
Technique: window function (RANK) to rank boroughs by superhost performance gap.

In [9]:
query2 = """
WITH borough_stats AS (
    SELECT
        neighbourhood_cleansed AS borough,
        host_is_superhost,
        COUNT(*) AS n_listings,
        ROUND(AVG(review_scores_rating), 3) AS avg_rating,
        ROUND(AVG(reviews_per_month), 2) AS avg_reviews_pm,
        ROUND(AVG(price), 2) AS avg_price
    FROM listings
    WHERE review_scores_rating IS NOT NULL
      AND host_is_superhost IS NOT NULL
    GROUP BY neighbourhood_cleansed, host_is_superhost
    HAVING COUNT(*) >= 30
)
SELECT
    borough,
    host_is_superhost,
    n_listings,
    avg_rating,
    avg_reviews_pm,
    avg_price,
    RANK() OVER (PARTITION BY host_is_superhost ORDER BY avg_rating DESC) AS rating_rank
FROM borough_stats
ORDER BY borough, host_is_superhost DESC
LIMIT 30
"""
pd.read_sql(query2, conn)

,borough,host_is_superhost,n_listings,avg_rating,avg_reviews_pm,avg_price,rating_rank
0,Barking and Dagenham,1,80,4.855,1.21,120.03,21
1,Barking and Dagenham,0,362,4.527,0.71,146.68,32
2,Barnet,1,361,4.859,1.37,167.80,19
3,Barnet,0,1332,4.601,0.73,164.89,24
4,Bexley,1,77,4.874,1.10,161.01,11
5,Bexley,0,356,4.682,0.61,137.02,12
6,Brent,1,562,4.861,1.48,215.94,18
7,Brent,0,1667,4.611,0.92,191.83,22
8,Bromley,1,155,4.886,1.26,156.88,5
9,Bromley,0,502,4.734,0.57,153.05,6


In [10]:
with open('../sql/query2_superhost_performance.sql', 'w') as f:
    f.write(query2)
print("Saved query2.sql")

Saved query2.sql


### Query 3 — Business Question 3 (groundwork)

Which listings generate the most review activity, and how does review volume relate to review scores? This query joins the listings and reviews tables, establishing the link required for the NLP sentiment analysis in Phase 7.

Columns used: listings.id, listings.neighbourhood_cleansed, listings.review_scores_rating, reviews.listing_id, reviews.date.
Technique: INNER JOIN across two tables with aggregation.

In [11]:
query3 = """
SELECT
    l.neighbourhood_cleansed AS borough,
    l.room_type,
    COUNT(DISTINCT l.id) AS n_listings,
    COUNT(r.id) AS total_reviews,
    ROUND(COUNT(r.id) * 1.0 / COUNT(DISTINCT l.id), 1) AS reviews_per_listing,
    ROUND(AVG(l.review_scores_rating), 3) AS avg_rating
FROM listings l
INNER JOIN reviews r
    ON l.id = r.listing_id
WHERE l.review_scores_rating IS NOT NULL
GROUP BY l.neighbourhood_cleansed, l.room_type
HAVING COUNT(DISTINCT l.id) >= 30
ORDER BY reviews_per_listing DESC
LIMIT 20
"""
pd.read_sql(query3, conn)

,borough,room_type,n_listings,total_reviews,reviews_per_listing,avg_rating
0,City of London,Private room,32,1629,50.9,4.684
1,Kensington and Chelsea,Private room,383,12576,32.8,4.723
2,Westminster,Shared room,36,1095,30.4,4.326
3,Camden,Private room,1075,32044,29.8,4.658
4,Islington,Private room,645,19109,29.6,4.775
5,Westminster,Private room,1130,33445,29.6,4.695
6,Southwark,Private room,1016,29379,28.9,4.762
7,Lambeth,Private room,997,28687,28.8,4.767
8,Richmond upon Thames,Private room,211,5755,27.3,4.892
9,Hammersmith and Fulham,Private room,541,13330,24.6,4.820


In [12]:
with open('../sql/query3_listings_reviews_join.sql', 'w') as f:
    f.write(query3)
print("Saved query3.sql")

Saved query3.sql
